# TC-WPN — Phase 2: the auxiliary-controlled ablation ladder

**Accelerator: GPU T4. Two training runs, ~95 minutes.**

This is your supervisor's section 35 and section 12, with one correction that
saves you a run.

## Experiment A is already done

Your supervisor asks for `protonet_temp_aux` = ProtoNet + temperature +
auxiliary. Checking `ABLATION_PRESETS`:

```
aux_only = (temporal=False, pcw=False, learn_temp=True, aux=0.3)
```

That is exactly ProtoNet + temperature + auxiliary. `aux_only` **is**
`protonet_temp_aux` — the same tuple, already trained, AUROC 0.73999.
Relabel it in the table rather than re-running it. A
`configs/protonet_temp_aux.yaml` alias exists if you prefer the name, but
running it would spend 48 minutes reproducing an identical model.

So Phase 2 needs **two** new runs, not three:

| config | tau | w^T | w^C | aux | status |
|---|:-:|:-:|:-:|:-:|---|
| `protonet` | ✗ | ✗ | ✗ | ✗ | done — 0.5223 |
| `protonet_temp` | ✓ | ✗ | ✗ | ✗ | done — 0.4985 |
| `temporal_pcw` | ✓ | ✓ | ✓ | ✗ | done — 0.4964 |
| `aux_only` *(= protonet_temp_aux)* | ✓ | ✗ | ✗ | ✓ | done — 0.7400 |
| **`temporal_aux`** | ✓ | ✓ | ✗ | ✓ | **run now** |
| **`pcw_aux`** | ✓ | ✗ | ✓ | ✓ | **run now** |
| `tcwpn_full` | ✓ | ✓ | ✓ | ✓ | done — 0.7335 |

These two yield Δ_temporal and Δ_PCW **with the auxiliary head held constant**,
which is the only way to attribute anything to w^T or w^C.

## What Phase 1 actually established

From `collapse_comparison.csv`:

| run | proto_cos | support_spread | gap_abs | p_sd | grad_encoder |
|---|---:|---:|---:|---:|---:|
| protonet | 0.99998 | 0.99989 | 0.00005 | 0.00016 | 1.39e-01 |
| protonet_temp | 0.99986 | 0.99935 | 0.00036 | 0.00083 | 3.27e-01 |
| pcw_only | 0.99984 | 0.99926 | 0.00032 | 0.00076 | 3.07e-01 |
| temporal_only | 0.99966 | 0.99869 | 0.00092 | 0.00182 | 1.73e-01 |
| **tcwpn_full** | **0.93159** | **0.90489** | **0.16311** | **0.23730** | **5.04e+00** |

One nuance worth stating precisely in the paper: the collapsed models are **not**
receiving zero gradient. Encoder gradient norms of 0.14–0.33 are small but not
vanishing — roughly 15–36× below tcwpn_full's 5.04. So the correct description
is *a degenerate optimum with a live but uninformative gradient*, not a dead
gradient path. Do not write "gradient vanishing"; the data does not say that.

## A gap this notebook closes

`stage_d_training_diagnostics.csv` shows `aux_only` with `loss_first` 0.9029,
`loss_last` 1.0509, `loss_min` 0.6998 — yet it reaches AUROC 0.74. That looks
contradictory until you notice the logged `loss` is the **total**
(proto + 0.3 × aux), so it cannot tell you whether the prototypical term ever
descended.

`train.py` now logs `proto_loss` and `aux_loss` separately. If `proto_loss`
sits at ~0.693 in `aux_only` while AUROC is 0.74, then the episodic objective
never learned and the prototypes are separable only because the auxiliary
gradient shaped the encoder. That is a sharp, quotable claim — and it is the
mechanism sentence of your paper.

In [ ]:
!rm -rf /kaggle/working/tcwpn_test
!git clone -q https://github.com/dulhara79/tcwpn_test.git /kaggle/working/tcwpn_test
%cd /kaggle/working/tcwpn_test
!pip install -q -r requirements.txt 2>&1 | tail -2

import subprocess, sys, os
r = subprocess.run([sys.executable, "-m", "pytest",
                    "tests/test_repo_layout.py", "tests/test_call_arity.py",
                    "-q", "--no-header"],
                   capture_output=True, text=True,
                   env={**os.environ, "PYTHONPATH": "src"})
print(r.stdout[-2000:])
if r.returncode != 0:
    raise SystemExit("Repository layout is broken — fix before spending GPU time.")

In [ ]:
from pathlib import Path
STAGE_A_DS = Path("/kaggle/input/datasets/dulharakaushalya/tc-wpn-stage-a-data")
STAGE_A = next((c for c in (STAGE_A_DS/"data"/"clean", STAGE_A_DS/"clean", STAGE_A_DS)
                if (c/"pkl").exists()), None)
if STAGE_A is None:
    raise SystemExit(f"no pkl/ under {STAGE_A_DS}")

PKL_DIR, PLAN_DIR = "/kaggle/working/pkl", str(STAGE_A/"plans")
STEM, K, SEED, RESULTS = "psych_mimic4idx", 5, 42, "/kaggle/working/results"
!mkdir -p {PKL_DIR}
!cp {STAGE_A}/pkl/*.pkl {PKL_DIR}/
print("ready")

## The two new runs

In [ ]:
for cfg in ["temporal_aux", "pcw_aux"]:
    print("="*70, f"\nTRAINING {cfg}\n", "="*70, sep="")
    !python -m scripts.train --config configs/{cfg}.yaml \
        --k {K} --seed {SEED} --stem {STEM} \
        --pkl-dir {PKL_DIR} --plan-dir {PLAN_DIR} --results {RESULTS}

## Regenerate `predictions_test.csv` for every model

Your Phase 1 run ended with

```
FileNotFoundError: .../tcwpn_full_k5_seed42/predictions_test.csv
```

because `tcwpn_full` was *diagnosed* from the Stage C input dataset but never
*re-evaluated* in that session, so no predictions file existed under
`/kaggle/working/results`. DeLong needs the paired score vectors, so every model
in the ladder must be evaluated in a session that can see its checkpoint.

The cell below copies any checkpoint it can find from `/kaggle/input` into the
working results tree first, then evaluates everything uniformly. Add your
Stage C and Phase 1 result datasets as inputs so the older checkpoints are
reachable.

In [ ]:
import os, glob, shutil

LADDER = ["protonet", "protonet_temp", "temporal_pcw", "aux_only",
          "temporal_aux", "pcw_aux", "tcwpn_full"]

for cfg in LADDER:
    name = f"{cfg}_k{K}_seed{SEED}"
    dst = f"{RESULTS}/{STEM}/{name}"
    if os.path.exists(f"{dst}/best.pt"):
        continue
    hits = glob.glob(f"/kaggle/input/**/{name}/best.pt", recursive=True)
    if hits:
        src = os.path.dirname(sorted(hits)[0])
        os.makedirs(dst, exist_ok=True)
        for f in os.listdir(src):
            shutil.copy2(os.path.join(src, f), dst)
        print(f"imported {cfg} from {src}")
    else:
        print(f"NOT FOUND: {cfg} — add the dataset containing it as an input")

print()
for cfg in LADDER:
    run = f"{RESULTS}/{STEM}/{cfg}_k{K}_seed{SEED}"
    if not os.path.exists(f"{run}/best.pt"):
        continue
    !python -m scripts.evaluate --run {run} --split test \
        --pkl-dir {PKL_DIR} --plan-dir {PLAN_DIR} --bootstrap 2000

In [ ]:
# Confirm every predictions file exists BEFORE running DeLong.
missing = [c for c in LADDER
           if not os.path.exists(f"{RESULTS}/{STEM}/{c}_k{K}_seed{SEED}/predictions_test.csv")]
print("ready:", [c for c in LADDER if c not in missing])
if missing:
    print("MISSING predictions_test.csv:", missing)
    print("DeLong will skip these pairs rather than crashing.")

## The ladder table

In [ ]:
import json, pandas as pd

rows = []
for cfg in LADDER:
    run = f"{RESULTS}/{STEM}/{cfg}_k{K}_seed{SEED}"
    ev, mf = f"{run}/eval_test.json", f"{run}/manifest.json"
    if not os.path.exists(ev):
        continue
    m = json.load(open(ev))["metrics"]
    row = {"model": cfg,
           "AUROC": round(m["auroc"], 4),
           "CI_low": round(m["auroc_ci_lower"], 4),
           "CI_high": round(m["auroc_ci_upper"], 4),
           "PR_AUC": round(m["pr_auc"], 4),
           "F1": round(m["f1_positive"], 4),
           "Sens": round(m["sensitivity"], 4),
           "Spec": round(m["specificity"], 4),
           "Brier": round(m["brier"], 4),
           "ECE": round(m["ece"], 4)}
    if os.path.exists(mf):
        h = json.load(open(mf)).get("history", [])
        if h:
            row["proto_loss_min"] = round(min(
                e.get("proto_loss", e["loss"]) for e in h), 4)
            row["aux_loss_min"] = (round(min(e["aux_loss"] for e in h), 4)
                                   if "aux_loss" in h[-1] else None)
    rows.append(row)

df = pd.DataFrame(rows).set_index("model")
print(df.to_string())
df.to_csv("/kaggle/working/phase2_ladder.csv")
print()
print("ln(2) = 0.6931. A proto_loss_min at ~0.693 in a model that still reaches")
print("AUROC ~0.74 means the episodic objective never learned and the auxiliary")
print("gradient did the representation work. That is the mechanism sentence.")

## Paired DeLong with Holm correction

Four comparisons, each isolating one thing. Holm rather than Bonferroni:
it is uniformly more powerful and equally valid.

In [ ]:
import subprocess, json
import pandas as pd

PAIRS = [
    ("protonet_temp", "aux_only",     "aux head, no mechanisms  -> delta_aux"),
    ("aux_only",      "temporal_aux", "add w^T, aux held constant -> delta_temporal"),
    ("aux_only",      "pcw_aux",      "add w^C, aux held constant -> delta_PCW"),
    ("aux_only",      "tcwpn_full",   "add both, aux held constant -> delta_interaction"),
]

out = []
for a, b, why in PAIRS:
    pa = f"{RESULTS}/{STEM}/{a}_k{K}_seed{SEED}/predictions_test.csv"
    pb = f"{RESULTS}/{STEM}/{b}_k{K}_seed{SEED}/predictions_test.csv"
    if not (os.path.exists(pa) and os.path.exists(pb)):
        print(f"skip {a} vs {b}: missing predictions"); continue
    dst = f"/kaggle/working/delong_{a}_vs_{b}.json"
    subprocess.run(["python", "-m", "scripts.compare_models", "pair",
                    "--a", pa, "--b", pb, "--out", dst], check=False)
    try:
        d = json.load(open(dst)); d["comparison"] = f"{a} -> {b}"; d["isolates"] = why
        out.append(d)
    except Exception as e:
        print("failed", a, b, e)

if out:
    t = pd.DataFrame(out).sort_values("p_value").reset_index(drop=True)
    m = len(t)
    t["holm_threshold"] = [0.05 / (m - i) for i in range(m)]
    t["significant"] = t["p_value"] <= t["holm_threshold"]
    # Holm is step-down: once one fails, all later ones fail too.
    first_fail = t.index[~t["significant"]].min() if (~t["significant"]).any() else None
    if first_fail is not None:
        t.loc[first_fail:, "significant"] = False
    cols = ["comparison", "isolates", "auroc_a", "auroc_b", "delta_auroc",
            "p_value", "holm_threshold", "significant"]
    print(t[[c for c in cols if c in t.columns]].to_string(index=False))
    t.to_csv("/kaggle/working/phase2_delong_holm.csv", index=False)

## Reading the result

Your supervisor set out the three outcomes in section 13. Match yours to one and
write the claim it licenses — nothing stronger.

**Mechanisms add nothing** (`temporal_aux` ≈ `pcw_aux` ≈ `aux_only` ≈ 0.74).
Then say so. The paper becomes: *episodic prototypical training collapses on
this task; a lightweight auxiliary supervised objective is necessary and
sufficient to prevent it; the proposed weightings do not add measurable value.*
That is a legitimate negative result with a mechanism, on a leakage-controlled
benchmark. It is publishable and your supervisor has explicitly endorsed it.

**Mechanisms add incrementally** (0.74 → 0.76 → 0.78 → 0.80). Then you have the
compositional contribution, and the DeLong column tells you whether each step is
real.

**TC-WPN below `aux_only`** — which is the *current* reading, 0.7335 vs 0.7400.
Then the mechanisms may be actively hurting, and that must be reported too.

Whatever happens, do not re-run with different seeds until a mechanism is
chosen. Section 32 of your supervisor's feedback lists that as the first thing
not to do, and it is the easiest mistake to make under deadline pressure.